In [13]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import random
from torch.utils.data import Dataset,DataLoader, TensorDataset
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight
import torch.nn.functional as F
from transformers import BertTokenizer, BertConfig, BertModel

SimpleNN

In [52]:
# fix random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Данные
df = pd.read_csv("df_key_rate.csv")
texts = list(df['text'])
labels = list(df['target'])

# TF-IDF векторы
vec = TfidfVectorizer(ngram_range=(1,3), max_features=5000)
X = vec.fit_transform(texts).toarray()

# переводим метки в 0,1,2
le = LabelEncoder()
y = le.fit_transform(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# переводим в тензоры
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# простая полносвязная модель
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden1=64, hidden2=32, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

model = SimpleNN(input_dim=X_train.shape[1])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# оптимизатор и loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# обучение
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

# предсказания и метрики
model.eval()
with torch.no_grad():
    logits = model(X_test_t.to(device))
    preds = torch.argmax(logits, dim=1).cpu()

print(classification_report(y_test, preds, digits=3))

f1_macro_simplenn1 = f1_score(y_test, preds, average='macro')
print(f"F1 macro: {f1_macro_simplenn1:.4f}")

Epoch 1/10, Loss: 1.1108
Epoch 2/10, Loss: 1.0928
Epoch 3/10, Loss: 1.0678
Epoch 4/10, Loss: 1.0279
Epoch 5/10, Loss: 0.9789
Epoch 6/10, Loss: 0.9154
Epoch 7/10, Loss: 0.8218
Epoch 8/10, Loss: 0.7545
Epoch 9/10, Loss: 0.6544
Epoch 10/10, Loss: 0.5907
              precision    recall  f1-score   support

           0      0.667     0.909     0.769        11
           1      0.500     0.600     0.545         5
           2      0.000     0.000     0.000         5

    accuracy                          0.619        21
   macro avg      0.389     0.503     0.438        21
weighted avg      0.468     0.619     0.533        21

F1 macro: 0.4382


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [53]:
results_df = pd.DataFrame(columns=["model", "f1_macro"])

results_df.loc[len(results_df)] = {
    "model": "SimpleNN слои: 64 и 32",
    "f1_macro": f1_macro_simplenn1
}
results_df

,model,f1_macro
0,SimpleNN слои: 64 и 32,0.438228


In [54]:
# fix random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# данные
df = pd.read_csv("df_key_rate.csv")
texts = list(df['text'])
labels = list(df['target'])

# TF-IDF векторы
vec = TfidfVectorizer(ngram_range=(1,3), max_features=5000)
X = vec.fit_transform(texts).toarray()

# переводим метки в 0,1,2
le = LabelEncoder()
y = le.fit_transform(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Переводим в тензоры
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# простая полносвязная модель
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden1=256, hidden2=128, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

model = SimpleNN(input_dim=X_train.shape[1])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# оптимизатор и loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# обучение
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

# предсказания и метрики
model.eval()
with torch.no_grad():
    logits = model(X_test_t.to(device))
    preds = torch.argmax(logits, dim=1).cpu()

print(classification_report(y_test, preds, digits=3))

f1_macro_simplenn2 = f1_score(y_test, preds, average='macro')
print(f"F1 macro: {f1_macro_simplenn2:.4f}")

Epoch 1/10, Loss: 1.0895
Epoch 2/10, Loss: 1.0200
Epoch 3/10, Loss: 0.9268
Epoch 4/10, Loss: 0.7673
Epoch 5/10, Loss: 0.6047
Epoch 6/10, Loss: 0.5183
Epoch 7/10, Loss: 0.3253
Epoch 8/10, Loss: 0.2309
Epoch 9/10, Loss: 0.1607
Epoch 10/10, Loss: 0.1228
              precision    recall  f1-score   support

           0      0.714     0.909     0.800        11
           1      0.600     0.600     0.600         5
           2      1.000     0.400     0.571         5

    accuracy                          0.714        21
   macro avg      0.771     0.636     0.657        21
weighted avg      0.755     0.714     0.698        21

F1 macro: 0.6571


In [55]:
results_df.loc[len(results_df)] = {
    "model": "SimpleNN слои: 256 и 128",
    "f1_macro": f1_macro_simplenn2
}
results_df

,model,f1_macro
0,SimpleNN слои: 64 и 32,0.438228
1,SimpleNN слои: 256 и 128,0.657143


In [56]:
# fix random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Данные
df = pd.read_csv("df_key_rate.csv")
texts = list(df['text'])
labels = list(df['target'])

# TF-IDF векторы
vec = TfidfVectorizer(ngram_range=(1,3), max_features=5000)
X = vec.fit_transform(texts).toarray()

# переводим метки в 0,1,2
le = LabelEncoder()
y = le.fit_transform(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Переводим в тензоры
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# простая полносвязная модель
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden1=256, hidden2=256, hidden3=256, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden2, hidden3)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(hidden3, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.fc4(x)
        return x

model = SimpleNN(input_dim=X_train.shape[1])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# оптимизатор и loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# обучение
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

# предсказания и метрики
model.eval()
with torch.no_grad():
    logits = model(X_test_t.to(device))
    preds = torch.argmax(logits, dim=1).cpu()

print(classification_report(y_test, preds, digits=3))

f1_macro_simplenn3 = f1_score(y_test, preds, average='macro')
print(f"F1 macro: {f1_macro_simplenn3:.4f}")

Epoch 1/10, Loss: 1.0974
Epoch 2/10, Loss: 1.0281
Epoch 3/10, Loss: 0.9311
Epoch 4/10, Loss: 0.8542
Epoch 5/10, Loss: 0.6338
Epoch 6/10, Loss: 0.4253
Epoch 7/10, Loss: 0.2583
Epoch 8/10, Loss: 0.1088
Epoch 9/10, Loss: 0.0585
Epoch 10/10, Loss: 0.0221
              precision    recall  f1-score   support

           0      0.750     0.818     0.783        11
           1      0.500     0.400     0.444         5
           2      0.800     0.800     0.800         5

    accuracy                          0.714        21
   macro avg      0.683     0.673     0.676        21
weighted avg      0.702     0.714     0.706        21

F1 macro: 0.6757


In [57]:
results_df.loc[len(results_df)] = {
    "model": "SimpleNN 3 слоя: 256",
    "f1_macro": f1_macro_simplenn3
}
results_df

,model,f1_macro
0,SimpleNN слои: 64 и 32,0.438228
1,SimpleNN слои: 256 и 128,0.657143
2,SimpleNN 3 слоя: 256,0.675684


AutoModelForSequenceClassification

In [58]:
# fix random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

df = pd.read_csv("df_key_rate.csv")

le = LabelEncoder()
df['target_enc'] = le.fit_transform(df['target'])

x_train, x_test, y_train, y_test = train_test_split(
    df.text, df.target_enc, random_state=42
)

# модель и токенайзер
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3,
    problem_type="single_label_classification")

# подготовка данных
train_encodings = tokenizer(list(x_train), truncation=True, padding=True, max_length=128, return_tensors="pt")
test_encodings  = tokenizer(list(x_test), truncation=True, padding=True, max_length=128, return_tensors="pt")

y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long)

train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], y_train)
test_dataset  = TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=8)

# forward pass для обучения
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 10

results = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

# предсказания
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=['-1','0','1']))

f1_macro_automodel = f1_score(all_labels, all_preds, average='macro')
print(f"F1 macro: {f1_macro_automodel:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

Epoch 1 | Loss: 1.1098
Epoch 2 | Loss: 1.0424
Epoch 3 | Loss: 0.9255
Epoch 4 | Loss: 0.8080
Epoch 5 | Loss: 0.5842
Epoch 6 | Loss: 0.3247
Epoch 7 | Loss: 0.2098
Epoch 8 | Loss: 0.0985
Epoch 9 | Loss: 0.0489
Epoch 10 | Loss: 0.0324
              precision    recall  f1-score   support

          -1       0.79      0.85      0.81        13
           0       0.70      0.88      0.78         8
           1       0.50      0.20      0.29         5

    accuracy                           0.73        26
   macro avg       0.66      0.64      0.63        26
weighted avg       0.70      0.73      0.70        26

F1 macro: 0.6261


In [59]:
results_df.loc[len(results_df)] = {
    "model": "AutoModelForSequenceClassification",
    "f1_macro": f1_macro_automodel
}
results_df

,model,f1_macro
0,SimpleNN слои: 64 и 32,0.438228
1,SimpleNN слои: 256 и 128,0.657143
2,SimpleNN 3 слоя: 256,0.675684
3,AutoModelForSequenceClassification,0.626102


In [26]:
# fix random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# загрузка данных
df = pd.read_csv("df_key_rate.csv")

# кодируем метки
le = LabelEncoder()
df['target_enc'] = le.fit_transform(df['target'])

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['target_enc'], test_size=0.2, random_state=42
)

# dataset
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# токенизатор
tokenizer = BertTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

# модель: BERT + BiLSTM + MLP + Attention Pooling
class BertBiLSTMMLP(nn.Module):
    def __init__(self, bert_model_name, hidden_size=128, lstm_hidden=64, num_labels=3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=lstm_hidden,
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(lstm_hidden*2, 64)
        self.fc2 = nn.Linear(64, num_labels)
        if self.bert.config.hidden_size != hidden_size:
            self.project = nn.Linear(self.bert.config.hidden_size, hidden_size)
        else:
            self.project = nn.Identity()

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        x = self.project(last_hidden)
        lstm_out, _ = self.lstm(x)
        x = torch.mean(lstm_out, dim=1)
        x = self.dropout(F.relu(self.fc1(x)))
        logits = self.fc2(x)

        return logits

# dataLoader
train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# устройство и оптимизатор
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertBiLSTMMLP("DeepPavlov/rubert-base-cased")
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

# обучение
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = F.cross_entropy(logits, labels, weight=class_weights.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_.astype(str), digits=3))

f1_macro_bertbilstm = f1_score(all_labels, all_preds, average='macro')
print(f"F1 macro: {f1_macro_bertbilstm:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Loss: 1.1016
Epoch 2 | Loss: 1.0927
Epoch 3 | Loss: 1.0939
Epoch 4 | Loss: 1.0643
Epoch 5 | Loss: 0.9766
Epoch 6 | Loss: 0.8523
Epoch 7 | Loss: 0.8131
Epoch 8 | Loss: 0.7162
Epoch 9 | Loss: 0.6962
Epoch 10 | Loss: 0.6839
              precision    recall  f1-score   support

        -1.0      1.000     0.818     0.900        11
         0.0      0.833     1.000     0.909         5
         1.0      0.833     1.000     0.909         5

    accuracy                          0.905        21
   macro avg      0.889     0.939     0.906        21
weighted avg      0.921     0.905     0.904        21

F1 macro: 0.9061


In [60]:
results_df.loc[len(results_df)] = {
    "model": "BertBiLSTMMLP",
    "f1_macro": f1_macro_bertbilstm
}
results_df

,model,f1_macro
0,SimpleNN слои: 64 и 32,0.438228
1,SimpleNN слои: 256 и 128,0.657143
2,SimpleNN 3 слоя: 256,0.675684
3,AutoModelForSequenceClassification,0.626102
4,BertBiLSTMMLP,0.906061


In [29]:
# random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# загрузка данных
df = pd.read_csv("df_key_rate.csv")

le = LabelEncoder()
df['target_enc'] = le.fit_transform(df['target'])

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['target_enc'], test_size=0.2, random_state=42
)

# dataset
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# токенизатор
tokenizer = BertTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

# модель: BERT + Attention Pooling + MLP
class BertAttentionClassifier(nn.Module):
    def __init__(self, bert_model_name, num_labels=3, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.attention = nn.Linear(self.bert.config.hidden_size, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = bert_out.last_hidden_state

        attn_weights = torch.softmax(self.attention(last_hidden), dim=1)
        pooled = torch.sum(attn_weights * last_hidden, dim=1)

        x = self.dropout(pooled)
        logits = self.fc(x)
        return logits

# dataLoader
train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# устройство и оптимизатор
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertAttentionClassifier("DeepPavlov/rubert-base-cased")
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-5)

# обучение
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = F.cross_entropy(logits, labels, weight=class_weights.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_.astype(str), digits=3))

f1_macro_bertattention = f1_score(all_labels, all_preds, average='macro')
print(f"F1 macro: {f1_macro_bertattention:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Loss: 1.2046
Epoch 2 | Loss: 1.0647
Epoch 3 | Loss: 0.9530
Epoch 4 | Loss: 0.6553
Epoch 5 | Loss: 0.3440
Epoch 6 | Loss: 0.2395
Epoch 7 | Loss: 0.1004
Epoch 8 | Loss: 0.0482
Epoch 9 | Loss: 0.0174
Epoch 10 | Loss: 0.0059
              precision    recall  f1-score   support

        -1.0      1.000     0.909     0.952        11
         0.0      1.000     1.000     1.000         5
         1.0      0.833     1.000     0.909         5

    accuracy                          0.952        21
   macro avg      0.944     0.970     0.954        21
weighted avg      0.960     0.952     0.953        21

F1 macro: 0.9538


In [61]:
results_df.loc[len(results_df)] = {
    "model": "BertAttention",
    "f1_macro": f1_macro_bertattention
}
results_df

,model,f1_macro
0,SimpleNN слои: 64 и 32,0.438228
1,SimpleNN слои: 256 и 128,0.657143
2,SimpleNN 3 слоя: 256,0.675684
3,AutoModelForSequenceClassification,0.626102
4,BertBiLSTMMLP,0.906061
5,BertAttention,0.953824


In [62]:
results_df.to_csv("results_df.csv", index=False)